# LLM Inference Experiments
### Temperature, Top-K and Top-P using the OpenRouter API


**What this cell does:**
- Downloads the three Python libraries we need
- `openai` → talks to OpenRouter (it uses the same format as OpenAI)
- `python-dotenv` → reads the API key from a `.env` file so it's not written in the notebook
- `pandas` → builds result tables later
- Only needed once per computer

## 1. Install libraries

In [1]:
# Install libraries (run once)
%pip install openai python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.


## 2. Import libraries

**What this cell does:**
- Loads the libraries we installed so we can use them in this notebook
- `os` → reads environment variables (like the API key)
- `time` → measures how long each response takes (latency)
- `pandas` → builds result tables
- `load_dotenv` → reads the API key from a `.env` file
- `OpenAI` → the client that sends prompts to OpenRouter
- The `print` line just confirms everything loaded

In [2]:
import os, time
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

print("All libraries imported ")

All libraries imported 


## 3.Get your OpenRouter API key

**Why we use a `.env` file:** the API key works like a password, so we keep it in a separate file instead of writing it in the notebook.

**Step 1: Create the key**
- Go to **openrouter** and sign in
- Click your profile (top right) → **Keys**
- Click **Create Key**, name it `llm-assignment`, you can select expirtion, key limit and reset limit accordingly and create it
- Copy the key immediately, because it is shown only once
- Never paste the key inside this notebook

**Step 2: Create the `.env` file**
- In the JupyterLab file panel (left), open the same folder as this notebook
- Click the blue **+** → under **Other**, click **Text File**
- Right-click the new file → **Rename** → name it exactly `.env`
- Type one line, with no spaces and no quotes: `OPENROUTER_API_KEY=your_key_here`
- Press **Ctrl + S** to save

**Note:** files starting with a dot are hidden by default. To see them, go to **View → Show Hidden Files**.

**What the below cell does:**
- Reads the `.env` file in this folder
- Gets the OpenRouter key from it
- Creates the `client` that sends prompts to OpenRouter
- Prints only a success message, never the key itself

In [3]:
load_dotenv()
api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    print("API key not found ❌ Check your .env file")
else:
    print("API key loaded ✅")

API key loaded ✅


## 4. First request to the model

**What this section does:**
- Creates a connection between the notebook and OpenRouter.
- Uses the API key loaded from the `.env` file.
- Sends a simple prompt to the selected LLM.
- Shows the model's response, token usage and response time.
- This is our first test to check whether the model is working correctly.

In [ ]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

print("Client created successfully ✅")

In [39]:
print("ask_model exists:", "ask_model" in globals())
print("MODEL:", MODEL)

ask_model exists: True
MODEL: nvidia/nemotron-3.5-lightning:free


**What this cell does:**
- Fixes one model (`MODEL`) so every experiment uses the same one
- Sends one prompt and measures how long the reply takes (latency)
- Prints the response and token usage
- No temperature, Top-K or Top-P yet, so the model uses its defaults

In [13]:
import requests

MODEL = "nvidia/nemotron-3.5-lightning:free"
prompt = "Write a Java method to check whether a number is even or odd."
headers = {"Authorization": f"Bearer {api_key}"}
body = {"model": MODEL, "max_tokens": 300, "reasoning": {"enabled": False},
        "messages": [{"role": "user", "content": prompt}]}

response = None
for attempt in range(1, 6):
    try:
        start = time.time()
        r = requests.post("https://openrouter.ai/api/v1/chat/completions",
                          headers=headers, json=body, timeout=60)
        latency = time.time() - start
        if r.status_code == 429:
            print(f"Rate limited (attempt {attempt}/5). Waiting 20 sec...")
            time.sleep(20)
            continue
        r.raise_for_status()
        response = r.json()
        
        break
    except requests.RequestException as e:
        print(f"Attempt {attempt}/5 failed: {type(e).__name__}")
        time.sleep(10)

if response is None:
    raise SystemExit("Failed after 5 attempts. Try again later or pick another model.")

print("Response:\n", response["choices"][0]["message"]["content"])
print("\nInput tokens :", response["usage"]["prompt_tokens"])
print("Output tokens:", response["usage"]["completion_tokens"])
print("Total tokens :", response["usage"]["total_tokens"])
print(f"Latency      : {latency:.2f} sec")

Response:
 Here is a simple Java method to check if a number is even or odd.

### The Method

```java
public class EvenOddChecker {

    // Method to check even or odd
    public static String checkEvenOdd(int number) {
        if (number % 2 == 0) {
            return "Even";
        } else {
            return "Odd";
        }
    }

    // Main method to test the function
    public static void main(String[] args) {
        int testNumber = 7;
        
        String result = checkEvenOdd(testNumber);
        System.out.println(testNumber + " is " + result);
    }
}
```

### How it works:
1.  **The Modulo Operator (`%`)**: This operator returns the remainder of a division.
2.  **The Logic**: If a number divided by 2 has a remainder of `0`, it is perfectly divisible by 2, meaning it is **Even**. If the remainder is `1` (or any non-zero value), it is **Odd**.

### Output:
```text
7 is Odd
```

Input tokens : 30
Output tokens: 227
Total tokens : 257
Latency      : 12.83 sec


## 5. Create a reusable function

**Why we create this function:**
- We will run the same model many times during our experiments.
- Instead of writing the API request again and again, we create one reusable function.
- The function allows us to change Temperature, Top-P and Top-K.
- It also records the number of tokens used and the response time.
- This makes the experiments easier to run and compare.

In [15]:
import requests

MODEL = "nvidia/nemotron-3.5-lightning:free"

def ask_model(prompt, temperature=None, top_p=None, top_k=None,
              max_tokens=300, retries=3):
    body = {"model": MODEL, "max_tokens": max_tokens,
            "reasoning": {"enabled": False},
            "messages": [{"role": "user", "content": prompt}]}
    if temperature is not None: body["temperature"] = temperature
    if top_p is not None: body["top_p"] = top_p
    if top_k is not None: body["top_k"] = top_k

    headers = {"Authorization": f"Bearer {api_key}"}

    for attempt in range(1, retries + 1):
        try:
            start = time.time()
            r = requests.post("https://openrouter.ai/api/v1/chat/completions",
                              headers=headers, json=body, timeout=60)
            latency = time.time() - start
            if r.status_code == 429:
                print(f"Rate limited ({attempt}/{retries}). Waiting 20 sec...")
                time.sleep(20)
                continue
            r.raise_for_status()
            data = r.json()
            return {
                "response": data["choices"][0]["message"]["content"],
                "input_tokens": data["usage"]["prompt_tokens"],
                "output_tokens": data["usage"]["completion_tokens"],
                "total_tokens": data["usage"]["total_tokens"],
                "latency_sec": round(latency, 2),
            }
        except requests.RequestException as e:
            print(f"Attempt {attempt} failed: {type(e).__name__}")
            time.sleep(10)
    return None

In [16]:
prompt = "Write a short explanation of Java exception handling."

result = ask_model(prompt, temperature=0.2)

print(result["response"])
print(result["input_tokens"], result["output_tokens"], result["total_tokens"], result["latency_sec"])

**Java Exception Handling** is a mechanism used to handle runtime errors so that the normal flow of the application can be maintained. It is a robust way to manage unexpected events that disrupt the normal execution of a program.

Here is a breakdown of the core concepts:

### 1. The Keywords
Java uses five keywords to handle exceptions:
*   **`try`**: Defines a block of code where an exception can occur.
*   **`catch`**: Catches the exception thrown in the `try` block and handles it.
*   **`finally`**: A block of code that runs **always**, regardless of whether an exception occurred or not (used for cleanup, like closing files).
*   **`throw`**: Explicitly throws an exception.
*   **`throws`**: Declares that a method might throw one or more exceptions, shifting the responsibility of handling them to the caller.

### 2. The Workflow
When an error occurs inside a `try` block, the normal execution stops. The Java Runtime System (JVM) creates an **Exception Object** and searches for a blo

## 6. Experiment 1 — Effect of Temperature

### What is Temperature?

Temperature controls how predictable or varied the model's response can be.

- **Low Temperature** → more predictable and consistent responses
- **High Temperature** → more varied and creative responses

In this experiment, we keep the model and prompt the same and change only the Temperature value.

This helps us observe how changing Temperature affects the generated response.

In [37]:
prompt = """
Describe a cat in a creative way.
"""

temperatures = [0.0, 0.5, 1.0, 2.0]

temperature_results = []

for temp in temperatures:
    result = ask_model(
        prompt,
        temperature=temp,
        max_tokens=100
    )

    if result is not None:
        temperature_results.append({
            "temperature": temp,
            "response": result["response"],
            "output_tokens": result["output_tokens"],
            "latency_sec": result["latency_sec"]
        })
    else:
        temperature_results.append({
            "temperature": temp,
            "response": "No response - rate limited",
            "output_tokens": None,
            "latency_sec": None
        })

df_temperature = pd.DataFrame(temperature_results)

df_temperature


,temperature,response,output_tokens,latency_sec
0,0.0,"She is a question mark made of fur, her tail a...",100,6.33
1,0.5,"She is a silhouette of smoke and shadow, a cre...",100,4.24
2,1.0,The cat moves like a question mark made of smo...,100,4.05
3,2.0,Listen. Can 제 prawa Eb \]\ndescribe оборow M 歴...,100,4.97


In [38]:
for _, row in df_temperature.iterrows():
    print("=" * 70)
    print(f"Temperature: {row['temperature']}")
    print("=" * 70)
    print(row["response"])
    print()

Temperature: 0.0
She is a question mark made of fur, her tail a question mark of pure intent, twitching with the anticipation of a hunt that exists only in her imagination. Her coat is the color of twilight settling over a city street, a sleek, midnight sable that seems to drink in the light rather than reflect it.

When she moves, it is with the liquid grace of a shadow detaching itself from a wall. There is no sound, only the faintest whisper of paws against hard

Temperature: 0.5
She is a silhouette of smoke and shadow, a creature of impossible grace who moves through the house like a ghost with nine lives already spent. Her fur is the color of twilight, a sleek obsidian that seems to drink the light rather than reflect it, punctuated only by a single, pristine white lock that sits atop her head like a mischievous crown.

When she walks, it is not a gait but a liquid flow; her paws barely kiss the floor, leaving no sound

Temperature: 1.0
The cat moves like a question mark made of s

In [ ]:
prompt = "Write a 2 lines story about a cat finding a mysterious box."

temperatures = [0.2, 0.5, 1.0]

temperature_results = []

for temp in temperatures:

    result = ask_model(
        prompt,
        temperature=temp,
        max_tokens=300
    )

    if result is not None and result["response"] is not None:

        temperature_results.append({
            "temperature": temp,
            "response": result["response"],
            "output_tokens": result["output_tokens"],
            "latency_sec": result["latency_sec"]
        })

    else:

        temperature_results.append({
            "temperature": temp,
            "response": "No text response received",
            "output_tokens": result["output_tokens"] if result else None,
            "latency_sec": result["latency_sec"] if result else None
        })

df_temperature = pd.DataFrame(temperature_results)

df_temperature

In [ ]:
for _, row in df_temperature.iterrows():

    print("=" * 70)
    print(f"Temperature: {row['temperature']}")
    print("=" * 70)
    print(row["response"])
    print()

## 7. Experiment 2 — TOP-P

To understand how Top-P controls the group of possible tokens from which the model can choose.

The model adds the probabilities of the most likely tokens and keeps selecting tokens until their combined probability reaches the Top-P value.

- **Low Top-P** → smaller group of possible tokens
- **High Top-P** → larger group of possible tokens

In this experiment, we keep the model, prompt and Temperature the same and change only Top-P.

In [31]:
prompt = "GIVE INTRODUCTION IN 3 LINES"

top_p_values = [0.2, 0.5, 1.0]
top_p_results = []

for top_p in top_p_values:
    result = ask_model(
        prompt,
        temperature=0.7,
        top_p=top_p,
        max_tokens=200
    )

    if result is not None and result["response"] is not None:
        top_p_results.append({
            "top_p": top_p,
            "response": result["response"],
            "output_tokens": result["output_tokens"],
            "latency_sec": result["latency_sec"]
        })
    else:
        top_p_results.append({
            "top_p": top_p,
            "response": "No text response received",
            "output_tokens": None,
            "latency_sec": None
        })

df_top_p = pd.DataFrame(top_p_results)
df_top_p[["top_p", "output_tokens", "latency_sec"]]

,top_p,output_tokens,latency_sec
0,0.2,61,14.68
1,0.5,17,2.06
2,1.0,79,4.57


In [32]:
for _, row in df_top_p.iterrows():

    print("=" * 70)
    print(f"Top-P: {row['top_p']}")
    print("=" * 70)
    print(row["response"])
    print()

Top-P: 0.2
I am Nemotron, a language model developed by NVIDIA. I can assist with a wide range of tasks, from answering questions to creative writing, all while prioritizing helpfulness and accuracy. My goal is to provide clear, concise, and useful information across various topics to assist users effectively.

Top-P: 0.5
I am Nemotron, a language model trained by researchers from NVIDIA.

Top-P: 1.0
I am Nemotron, a language model trained by researchers from NVIDIA. I can help with a wide variety of tasks, from answering questions to creative writing, and I'm designed to understand context and provide accurate information across many topics. Whether you need assistance with coding, analysis, or just have a general question, I'm here to help you find the information you're looking for.



## 8. Experiment 3 — Top-K

Top-K tells the model to consider only the **K most probable tokens** when choosing the next token.

For example:

- **Top-K = 1** → considers only the most probable token
- **Top-K = 10** → considers the 10 most probable tokens
- **Top-K = 100** → considers the 100 most probable tokens

In this experiment, we keep the model, prompt and Temperature the same and change only Top-K.

In [33]:
prompt = "Write a short description of an umbrella."

top_k_values = [1, 10, 100]
top_k_results = []

for top_k in top_k_values:
    result = ask_model(prompt, temperature=0.7, top_k=top_k, max_tokens=200)

    if result is not None and result["response"] is not None:
        top_k_results.append({
            "top_k": top_k,
            "response": result["response"],
            "output_tokens": result["output_tokens"],
            "latency_sec": result["latency_sec"]
        })
    else:
        top_k_results.append({
            "top_k": top_k,
            "response": "No text response received",
            "output_tokens": None,
            "latency_sec": None
        })

df_top_k = pd.DataFrame(top_k_results)
df_top_k[["top_k", "output_tokens", "latency_sec"]]

,top_k,output_tokens,latency_sec
0,1,91,5.56
1,10,85,3.48
2,100,88,3.91


In [36]:
for _, row in df_top_k.iterrows():
    print("=" * 70)
    print(f"Top-K: {row['top_k']}")
    print("=" * 70)
    print(row["response"])
    print()

Top-K: 1
A faithful companion against the elements, the umbrella is a portable canopy designed to shield its bearer from rain or harsh sunlight. Typically consisting of a collapsible metal or wooden pole, a curved handle for a secure grip, and a fabric canopy stretched over ribs, it folds compactly for easy transport. Whether shielding a commuter from a sudden downpour or providing shade on a sunny beach, the umbrella remains a simple, indispensable tool of daily life.

Top-K: 10
A reliable companion against the elements, an umbrella is a portable canopy designed to shield its bearer from rain or harsh sunlight. Typically featuring a collapsible metal or wooden frame and a fabric canopy, it opens with a satisfying snap to create a dry sanctuary. Whether crafted in a sleek, modern design or a vibrant, patterned print, the umbrella remains an essential tool for navigating the unpredictability of the weather in style.

Top-K: 100
A trusty companion against the elements, the umbrella is a 